<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=350095850" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PPE YOLOv8 training (Colab / Kaggle)

Use **this notebook** for E0–E4. Local 8GB GPUs are only for baseline val, export, and `--batch 8` smokes.

Product bars: **vest / no_vest 95%+**, helmets next, goggles ~70% OK. **Boots are not in this cycle.**

1. Runtime → GPU (Colab) or GPU accelerator (Kaggle).
2. Add secret `ROBOFLOW_API_KEY` (Colab userdata / Kaggle Add-ons → Secrets).
3. Run all cells. Default experiment: `e0_n` on the 12k subset after Combined download + remap.

Prefer **one** of Colab or Kaggle, not both.

In [1]:
import os
from pathlib import Path

# Colab secret, then Kaggle, then env (local fallback).
try:
    from google.colab import userdata
    os.environ.setdefault("ROBOFLOW_API_KEY", userdata.get("ROBOFLOW_API_KEY"))
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("ROBOFLOW_API_KEY", UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    except Exception:
        pass

assert os.environ.get("ROBOFLOW_API_KEY"), "Set ROBOFLOW_API_KEY as a Colab/Kaggle secret"
print("key_set", "colab" if IN_COLAB else "kaggle_or_local")

REPO = Path("/content/ppe") if IN_COLAB else Path("/kaggle/working/ppe")
if not (REPO / "scripts" / "train.py").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git {REPO}
os.chdir(REPO)
print("cwd", Path.cwd())

key_set kaggle_or_local
Cloning into '/kaggle/working/ppe'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 148 (delta 2), reused 114 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 49.75 MiB | 33.69 MiB/s, done.
Resolving deltas: 100% (2/2), done.
cwd /kaggle/working/ppe


In [2]:
%pip install -q ultralytics roboflow pyyaml opencv-python-headless
%pip install -q -e .
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ppe (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
cuda True Tesla T4


In [3]:
# Combined (~2.4GB zip) then Hard Hat Universe. Construction is optional for mapped eval.
!python scripts/download_datasets.py --execute --only combined hardhat
!python scripts/remap_labels.py --source data/raw/combined --out data/processed/combined --mapping combined
!python scripts/remap_labels.py --source data/raw/hardhat --out data/processed/hardhat --mapping hhu
!python scripts/make_subset.py --source data/processed/combined --out data/raw/combined_12k --n 12000 --seed 42
!python scripts/analyze_distribution.py

Requesting export roboflow-universe-projects/personal-protective-equipment-combined-model/4/yolov8 -> /kaggle/working/ppe/data/raw/combined
  zip 100.0% [2542363062/2542363062 bytes]
Extracting combined.zip -> /kaggle/working/ppe/data/raw/combined
Downloaded combined to /kaggle/working/ppe/data/raw/combined
Hard Hat Universe: using version 26 (prefer 26 no_nulls_plain)
Requesting export universe-datasets/hard-hat-universe-0dy7t/26/yolov8 -> /kaggle/working/ppe/data/raw/hardhat
  zip 100.0% [245677789/245677789 bytes]
Extracting hardhat.zip -> /kaggle/working/ppe/data/raw/hardhat
Downloaded hardhat to /kaggle/working/ppe/data/raw/hardhat
train: 30765 images, dropped 0 unmapped boxes
valid: 8814 images, dropped 0 unmapped boxes
test: 4423 images, dropped 0 unmapped boxes
Wrote remapped dataset to /kaggle/working/ppe/data/processed/combined
train: 4912 images, dropped 0 unmapped boxes
valid: 1414 images, dropped 0 unmapped boxes
test: 708 images, dropped 0 unmapped boxes
Wrote remapped da

In [4]:
# E0 on 12k. Swap --exp: e1_s | e2_focal | e3_augs | e4_full44k
# P100/T4: batch 16. If OOM, add --batch 8.
!python scripts/train.py --exp e0_n --device 0 --batch 16

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Experiment: e0_n
Config:     /kaggle/working/ppe/configs/train/e0_n.yaml
Model:      yolov8n.pt
Train kwargs:
  batch: 16
  close_mosaic: 10
  cos_lr: True
  data: /kaggle/working/ppe/configs/data/combined.yaml
  device: 0
  epochs: 100
  exist_ok: True
  imgsz: 640
  name: e0_n
  optimizer: auto
  patience: 20
  pretrained: True
  project: /kaggle/working/ppe/runs/train
  seed: 42
Ultralytics 8.4.153 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, c

After E0, copy `runs/train/e0_n/weights/best.pt` off the VM (Drive / Kaggle output). Then run `eval.py` / `calibrate.py` and lead the report with **vest / no_vest**.